# Is That Isekai Tower Too High?

In isekai fantasy, the hero often arrives in a medieval-looking town and spots an impossibly tall tower on the horizon. But how tall is *too* tall? This notebook checks whether a tower's height is **visually plausible** — not structurally, but whether it would look that tall to the human eye given viewing distance, human field of view, and planetary curvature.

## The Sight-Line Problem (Flat Earth Approximation)

When you stand at horizontal distance $D$ from a tower of height $H$, the elevation angle $\theta$ (neck tilt) to see the top is:

$$\theta = \arctan\left(\frac{H}{D}\right)$$

This works well at medieval town scales (hundreds of meters), but breaks down at longer anime-scale distances.

## With Planetary Curvature

On a spherical planet of radius $R$, an observer's eye at height $h_e$ and a tower top at height $H$ separated by arc distance $D$ subtend angle $\alpha = D/R$. The elevation angle above the local horizontal is:

$$\theta = \operatorname{atan2}\bigl((R+H)\cos\alpha - (R+h_e),\;(R+H)\sin\alpha\bigr)$$

Earth's curvature also hides the base of distant objects — the **bulge drop** at distance $D$ is approximately $D^2 / 2R$, which makes far-away towers appear shorter than a flat-earth model predicts.

If $\theta$ exceeds what a human can comfortably look upward, the tower is "too high" for that viewing distance.

In [ ]:
import math

# --- Planet ---
EARTH_RADIUS_M = 6_371_000.0
OBSERVER_EYE_HEIGHT_M = 1.7  # average standing eye height

# --- Human FOV / neck comfort thresholds (degrees above horizontal) ---
VERTICAL_FOV_TOTAL_DEG = 150.0
UPWARD_GAZE_NO_HEAD_MOVEMENT_DEG = 20.0
COMFORTABLE_NECK_TILT_DEG = 15.0
ERGONOMIC_UPPER_BOUND_DEG = 45.0
MAX_UPWARD_HEAD_TILT_DEG = 65.0

# --- Medieval town scale (meters) ---
TYPICAL_BASTIDE_DIAMETER_M = 500.0
LARGE_WALLED_CITY_DIAMETER_M = 1600.0
WALKING_DISTANCE_TOWN_LIMIT_M = 1000.0


def neck_tilt_degrees(distance_m: float, height_m: float) -> float:
    """Elevation angle (degrees) using flat-earth approximation."""
    return math.degrees(math.atan(height_m / distance_m))


def neck_tilt_degrees_curved(
    distance_m: float,
    height_m: float,
    eye_height_m: float = OBSERVER_EYE_HEIGHT_M,
    planet_radius_m: float = EARTH_RADIUS_M,
) -> float:
    """Elevation angle (degrees) accounting for planetary curvature."""
    alpha = distance_m / planet_radius_m
    dx = (planet_radius_m + height_m) * math.cos(alpha) - (planet_radius_m + eye_height_m)
    dy = (planet_radius_m + height_m) * math.sin(alpha)
    return math.degrees(math.atan2(dx, dy))


def curvature_drop_m(distance_m: float, planet_radius_m: float = EARTH_RADIUS_M) -> float:
    """Approximate vertical drop due to Earth's bulge at distance D."""
    return distance_m ** 2 / (2 * planet_radius_m)


def height_from_tilt(distance_m: float, tilt_deg: float) -> float:
    """Tower height implied by distance and neck tilt (flat-earth)."""
    return distance_m * math.tan(math.radians(tilt_deg))


def distance_from_tilt(height_m: float, tilt_deg: float) -> float:
    """Viewing distance implied by tower height and neck tilt (flat-earth)."""
    return height_m / math.tan(math.radians(tilt_deg))


def is_tower_too_high(distance_m: float, height_m: float, use_curvature: bool = True) -> str:
    """Verdict based on neck tilt vs human FOV comfort thresholds."""
    if use_curvature:
        tilt = neck_tilt_degrees_curved(distance_m, height_m)
    else:
        tilt = neck_tilt_degrees(distance_m, height_m)
    if tilt <= COMFORTABLE_NECK_TILT_DEG:
        return "comfortable"
    if tilt <= ERGONOMIC_UPPER_BOUND_DEG:
        return "uncomfortable"
    if tilt <= MAX_UPWARD_HEAD_TILT_DEG:
        return "strained"
    return "too high"


# Example: 100 m tower at various distances — flat vs curved
tower_height_m = 100.0
print(f"{'Distance (m)':>14}  {'Flat (°)':>10}  {'Curved (°)':>12}  {'Drop (m)':>10}  {'Verdict':>16}")
print("-" * 68)
for distance_m in [200, 500, 1000, 5000, 20000]:
    flat = neck_tilt_degrees(distance_m, tower_height_m)
    curved = neck_tilt_degrees_curved(distance_m, tower_height_m)
    drop = curvature_drop_m(distance_m)
    verdict = is_tower_too_high(distance_m, tower_height_m)
    print(f"{distance_m:>14.0f}  {flat:>10.1f}  {curved:>12.1f}  {drop:>10.1f}  {verdict:>16}")

## Remarks: Human FOV, Planetary Curvature, and Medieval Town Scale

### Average Human Field of View

- [**Field of view (Wikipedia)**](https://en.wikipedia.org/wiki/Field_of_view): The vertical visual field in humans is around **150°** total.
- **Upward gaze without head movement**: roughly **20°** above the horizontal line of sight.
- **Comfortable neck tilt** for extended viewing: about **15°** (no subjective fatigue).
- **Ergonomic upper bound**: around **45°** before sustained discomfort.
- **Maximum upward head tilt** (with neck rotation): roughly **65°** — beyond this, seeing the top of a tower becomes physically impractical.

### Planetary Curvature

- Earth mean radius: **6,371 km**.
- At distance $D$, the ground drops by roughly $D^2 / 2R$ — only **~2 cm** at 500 m, but **~2 m** at 5 km and **~31 m** at 20 km.
- Curvature makes distant towers appear **shorter** than a flat-earth model predicts, and can hide the base entirely beyond the horizon (~5 km for a standing observer).
- At medieval town scales (< 1 km), flat and curved models agree to within a fraction of a degree. At anime-scale distances (tens of km), curvature is essential for judging visual realism.

### Average Medieval City / Town Size

- **Typical bastide / fortified town**: defensive perimeter around **500 m** in diameter (~10 hectares).
- **Large walled city**: roughly **0.5–1 sq mi** (~800–1600 m across within the walls).
- **Walking-distance constraint**: most settlements stayed under **~1 km** across because daily life required everything to be within reasonable walking distance.

Standing at a town wall (~250 m from center) or just outside (~500 m), a tower in the town center must stay short enough that your neck tilt stays within the comfort zone — otherwise, that isekai tower really is too high.

## Create and Display the Plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

try:
    plt.style.use('seaborn-v0_8-darkgrid')
except OSError:
    try:
        plt.style.use('seaborn-darkgrid')
    except OSError:
        plt.style.use('default')
        print("Using default matplotlib style")

fig, ax = plt.subplots(figsize=(12, 8))

distances_m = np.linspace(50, 20000, 500)
tower_heights_m = [50, 100, 300, 1000]
colors = ['#4CAF50', '#2196F3', '#FF9800', '#E91E63']

for height_m, color in zip(tower_heights_m, colors):
    tilts_curved = np.array([neck_tilt_degrees_curved(d, height_m) for d in distances_m])
    ax.plot(distances_m, tilts_curved, linewidth=2.5, color=color,
            label=f'{height_m} m tower (curved)')

# Flat-earth reference for tallest tower
height_ref = 1000
tilts_flat = np.degrees(np.arctan(height_ref / distances_m))
ax.plot(distances_m, tilts_flat, linewidth=1.5, color='#E91E63', linestyle='--', alpha=0.5,
        label=f'{height_ref} m tower (flat earth)')

# Comfort zones (horizontal bands)
ax.axhspan(0, COMFORTABLE_NECK_TILT_DEG, alpha=0.12, color='green',
           label=f'Comfortable (0–{COMFORTABLE_NECK_TILT_DEG:.0f}°)')
ax.axhspan(COMFORTABLE_NECK_TILT_DEG, ERGONOMIC_UPPER_BOUND_DEG, alpha=0.10, color='orange',
           label=f'Uncomfortable ({COMFORTABLE_NECK_TILT_DEG:.0f}–{ERGONOMIC_UPPER_BOUND_DEG:.0f}°)')
ax.axhspan(ERGONOMIC_UPPER_BOUND_DEG, MAX_UPWARD_HEAD_TILT_DEG, alpha=0.08, color='red',
           label=f'Strained ({ERGONOMIC_UPPER_BOUND_DEG:.0f}–{MAX_UPWARD_HEAD_TILT_DEG:.0f}°)')
ax.axhline(MAX_UPWARD_HEAD_TILT_DEG, color='darkred', linestyle='--', linewidth=1.5,
           label=f'Max head tilt ({MAX_UPWARD_HEAD_TILT_DEG:.0f}°)')

# Medieval town scale reference lines
town_refs = {
    'Town wall (~250 m)': TYPICAL_BASTIDE_DIAMETER_M / 2,
    'Bastide diameter (500 m)': TYPICAL_BASTIDE_DIAMETER_M,
    'Walking limit (1 km)': WALKING_DISTANCE_TOWN_LIMIT_M,
}
for label, dist in town_refs.items():
    ax.axvline(dist, color='gray', linestyle=':', linewidth=1.2, alpha=0.7)
    ax.text(dist + 30, 72, label, fontsize=8, color='gray', rotation=90, va='top')

# Horizon distance for standing observer
horizon_m = math.sqrt(2 * EARTH_RADIUS_M * OBSERVER_EYE_HEIGHT_M)
ax.axvline(horizon_m, color='purple', linestyle='-.', linewidth=1.2, alpha=0.7)
ax.text(horizon_m + 30, 5, f'Horizon (~{horizon_m/1000:.1f} km)', fontsize=8, color='purple', rotation=90, va='bottom')

ax.set_xlim(50, 20000)
ax.set_ylim(0, 80)
ax.set_xlabel('Distance to Building (m)', fontsize=12, fontweight='bold')
ax.set_ylabel('Neck Tilt to See Top (°)', fontsize=12, fontweight='bold')
ax.set_title('Is That Isekai Tower Too High?\n'
             'Neck Tilt vs Viewing Distance (with Planetary Curvature)',
             fontsize=14, fontweight='bold', pad=20)
ax.grid(True, alpha=0.2)
ax.legend(loc='upper right', fontsize=8, framealpha=0.9)

source_text = (
    'References:\n'
    '• Human vertical FOV ~150° (Wikipedia)\n'
    '• Earth radius 6,371 km\n'
    '• Medieval bastide ~500 m diameter'
)
ax.text(0.02, 0.55, source_text, transform=ax.transAxes,
        fontsize=8, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.7))

plt.tight_layout()

output_path = Path('isekai_tower_too_high.png')
plt.savefig(output_path, dpi=300, bbox_inches='tight')
print(f'Plot saved to: {output_path.absolute()}')

plt.show()